# Curriculum Generator Using LangGraph
### Final Project for the course Introduction to Large Language Models (MAT496)

First, we import all the necessary libraries

In [1]:
%%capture --no-stderr
# If needed (run once in your environment)
# %pip install -U langgraph langchain groq

Setup the environment

In [2]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("GROQ_API_KEY")

 Setting up the call function to our desired model 

 I am using the `llama-3.1-8b-instant` model from Groq

In [3]:
from groq import Groq

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("Please set GROQ_API_KEY in your environment.")

groq_client = Groq(api_key=GROQ_API_KEY)

MODEL_NAME = "llama-3.1-8b-instant"  # or any other Groq model you want

def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.3) -> str:
    """
    Thin wrapper around Groq chat.completions.
    Keeps prompts small and structured to work well with lighter models.
    """
    completion = groq_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )
    return completion.choices[0].message.content.strip()

## CurriculumState Structure

 All information flows through a single structured dictionary:

 User Input:  
 `user_goal`, `level`, `duration_weeks`, `hours_per_week`

 Inferred Information:  
 `refined_goal`, `subject_tag`

 Intermediate Outputs:  
 `outline`, `text_resources`, `video_resources`, `course_resources`

 Final Output:  
 `final_plan` (the complete weekly curriculum)

 This strict structure ensures consistency across the entire pipeline.

In [4]:
from typing import List, Optional, Literal, Dict
from typing_extensions import TypedDict

class CurriculumState(TypedDict, total=False):
    # User input / normalized config
    user_goal: str
    level: Literal["beginner", "intermediate", "advanced"]
    duration_weeks: int
    hours_per_week: int
    
    # Inferred / classified
    subject_tag: str          # e.g. "ece", "ml", "programming", ...
    refined_goal: str         # more detailed version of user_goal
    
    # Intermediate artifacts
    outline: str              # high-level week / topic outline
    
    text_resources: List[str]     # book/articles
    video_resources: List[str]    # videos/playlists
    course_resources: List[str]   # MOOCs / structured courses
    
    # Final product
    final_plan: str           # fully formatted curriculum text


## Node-by-Node Breakdown
#### 1. Input Normalization (`normalize_input`)
Ensures clean defaults and consistent data types.  

This prevents errors during the graph run.

In [5]:
def normalize_input(
    user_goal: str,
    level: Optional[str] = None,
    duration_weeks: Optional[int] = None,
    hours_per_week: Optional[int] = None,
) -> CurriculumState:
    """
    Create an initial CurriculumState with defaults.
    This is NOT a graph node; you'll use this before app.invoke().
    """
    print("NORMALIZE INPUT RUNNING.....")
    if level not in {"beginner", "intermediate", "advanced"}:
        level = "beginner"

    if duration_weeks is None or duration_weeks <= 0:
        duration_weeks = 8  # default 8-week plan

    if hours_per_week is None or hours_per_week <= 0:
        hours_per_week = 5  # default 5 hrs/week

    return CurriculumState(
        user_goal=user_goal,
        level=level,  # type: ignore
        duration_weeks=duration_weeks,
        hours_per_week=hours_per_week,
    )


#### 2. Goal Expansion (`expand_goal_node`)

The model rewrites the initial user goal into a precise, clear, and focused description.

This improves the quality of all later steps.

In [6]:
def expand_goal_node(state: CurriculumState) -> Dict:
    """
    Use the LLM to clarify the goal and infer missing details (within reason).
    Keeps output small & structured for lightweight models.
    """
    print("EXPAND GOAL NODE RUNNING.....")
    
    system_prompt = (
        "You help clarify a student's learning goal for building a curriculum. "
        "Be concise and avoid long essays."
    )
    user_prompt = f"""
User goal: {state.get('user_goal')}

User level: {state.get('level')}
Duration (weeks): {state.get('duration_weeks')}
Hours per week: {state.get('hours_per_week')}

1. Rewrite the learning goal in 2–3 clear sentences.
2. If needed, infer a slightly more precise focus (e.g., 'digital design with Verilog for FPGAs' instead of just 'digital design').
3. Do NOT add requirements the user clearly didn't imply.
Output only the rewritten goal text.
"""
    refined = call_llm(system_prompt, user_prompt, temperature=0.2)
    return {"refined_goal": refined}

#### 3. Subject Classification (`route_subject_node`)

The refined learning goal is mapped to one tag:

- `ece`

- `programming`

- `ml`

- `math`

- `data`

- `career`

- `other`

This tag helps customize resource suggestions.

In [7]:
def route_subject_node(state: CurriculumState) -> Dict:
    """
    Classify the refined goal into a coarse subject tag.
    Lightweight models are usually good at this kind of classification.
    """
    print("ROUTE SUBJECT NODE RUNNING.....")

    system_prompt = (
        "You classify learning goals into a short subject tag. "
        "Return ONLY ONE lowercase tag from this set:\n"
        "['programming', 'ece', 'ml', 'math', 'data', 'career', 'other']"
    )
    user_prompt = f"Learning goal: {state.get('refined_goal') or state.get('user_goal')}"
    
    tag = call_llm(system_prompt, user_prompt, temperature=0.0)
    tag = tag.strip().lower()
    if tag not in {"programming", "ece", "ml", "math", "data", "career", "other"}:
        tag = "other"
    return {"subject_tag": tag}


#### 4. Curriculum Outline Generation (`generate_outline_node`)

Creates a week-by-week outline, matching:

- `Duration`

- `Level`

- `Realistic progression`

- `Proper weekly workload`

The output contains themes and bullet points for each week.

In [8]:
def generate_outline_node(state: CurriculumState) -> Dict:
    """
    Create a week-wise outline based on refined goal, subject, duration and level.
    This is still high-level (topics only).
    """
    print("GENERATE OUTLINE NODE RUNNING.....")

    system_prompt = (
        "You are a curriculum designer. "
        "Create a week-wise topic outline for the learning goal."
    )
    user_prompt = f"""
Learning goal (refined): {state.get('refined_goal')}
Subject tag: {state.get('subject_tag')}
Level: {state.get('level')}
Duration: {state.get('duration_weeks')} weeks
Hours per week: {state.get('hours_per_week')}

Requirements:
- Use exactly {state.get('duration_weeks')} weeks.
- For each week, give: 'Week N: <theme>' and 3–5 bullet topics.
- Keep topics concise (no paragraphs).
- Focus on a realistic progression for this level and time.
"""
    outline = call_llm(system_prompt, user_prompt, temperature=0.3)
    return {"outline": outline}


#### 5. Parallel Resource Gathering

Three independent nodes run simultaneously:

- `gather_text_resources_node`
- `gather_video_resources_node`
- `gather_course_resources_node`

Each produces:
- Books & articles
- Video lectures/playlists
- MOOC-style structured courses

Parallel execution is one of the strengths of LangGraph.

In [9]:
def gather_text_resources_node(state: CurriculumState) -> Dict:
    """
    Suggest books / articles / documentation.
    For smaller models, keep it short and template-like.
    """
    print("GATHER TEXT RESOURCES NODE RUNNING.....")

    system_prompt = (
        "You suggest high-level learning resources (books, docs, blogs). "
        "Use generic but realistic-sounding resource names (the user will verify externally)."
    )
    user_prompt = f"""
Learning goal: {state.get('refined_goal')}
Subject: {state.get('subject_tag')}
Outline:
{state.get('outline')}

Suggest 5–8 key text-based resources:
- Some foundational references
- Some topic-specific docs or tutorials
Format as a simple numbered list with short one-line descriptions.
"""
    resources = call_llm(system_prompt, user_prompt, temperature=0.4)
    # split into list of lines, filter blanks
    lines = [line.strip() for line in resources.split("\n") if line.strip()]
    return {"text_resources": lines}


def gather_video_resources_node(state: CurriculumState) -> Dict:
    """
    Suggest video resources (YouTube playlists, lecture series, etc.).
    """
    print("GATHER VIDEO RESOURCES NODE RUNNING.....")

    system_prompt = (
        "You recommend video-based learning resources (YouTube playlists, lecture series). "
        "Use generic but realistic-sounding titles."
    )
    user_prompt = f"""
Learning goal: {state.get('refined_goal')}
Subject: {state.get('subject_tag')}
Outline:
{state.get('outline')}

Suggest 3–6 main video resources.
Format as a numbered list: title + what it covers.
"""
    resources = call_llm(system_prompt, user_prompt, temperature=0.4)
    lines = [line.strip() for line in resources.split("\n") if line.strip()]
    return {"video_resources": lines}


def gather_course_resources_node(state: CurriculumState) -> Dict:
    """
    Suggest structured MOOCs/courses.
    """
    print("GATHER COURSE RESOURCES NODE RUNNING.....")

    system_prompt = (
        "You recommend structured online courses (MOOCs, specializations). "
        "Use generic platform names like 'MOOC platform' or 'Online course'."
    )
    user_prompt = f"""
Learning goal: {state.get('refined_goal')}
Subject: {state.get('subject_tag')}
Outline:
{state.get('outline')}

Suggest 3–5 structured courses, ordered from beginner to advanced.
Format as a numbered list with 1–2 lines each.
"""
    resources = call_llm(system_prompt, user_prompt, temperature=0.4)
    lines = [line.strip() for line in resources.split("\n") if line.strip()]
    return {"course_resources": lines}


#### 6. Curriculum Assembly (`build_curriculum_node`)

This node merges all components into a final polished plan:

- Weekly themes

- Specific learning topics

- 1–3 curated resources per week

- Clean, actionable structure

- This is the “reduce” stage of the map-reduce pattern.

In [10]:
def build_curriculum_node(state: CurriculumState) -> Dict:
    """
    Combine outline + all resources into a full curriculum description.
    This is the 'reduce' step of the map-reduce pattern.
    """
    print("BUILD CURRICULUM NODE RUNNING.....")

    system_prompt = (
        "You build a clear curriculum for a learner based on an outline and resources."
    )
    user_prompt = f"""
Learning goal: {state.get('refined_goal')}
Level: {state.get('level')}
Duration: {state.get('duration_weeks')} weeks
Hours per week: {state.get('hours_per_week')}
Subject tag: {state.get('subject_tag')}

=== OUTLINE ===
{state.get('outline')}

=== TEXT RESOURCES ===
{chr(10).join(state.get('text_resources', []))}

=== VIDEO RESOURCES ===
{chr(10).join(state.get('video_resources', []))}

=== COURSE RESOURCES ===
{chr(10).join(state.get('course_resources', []))}

TASK:
- Create a final plan organised by week.
- For each week: theme, key topics, and 1–3 suggested resources (mix of text/video/course).
- Do NOT invent very detailed URLs; keep resource names simple.
- Keep the whole curriculum concise but actionable.
"""
    plan = call_llm(system_prompt, user_prompt, temperature=0.35)
    return {"final_plan": plan}


### Graph Architecture

In [11]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(CurriculumState)

# Nodes
builder.add_node("expand_goal", expand_goal_node)
builder.add_node("route_subject", route_subject_node)
builder.add_node("generate_outline", generate_outline_node)

builder.add_node("gather_text_resources", gather_text_resources_node)
builder.add_node("gather_video_resources", gather_video_resources_node)
builder.add_node("gather_course_resources", gather_course_resources_node)

builder.add_node("build_curriculum", build_curriculum_node)

# Edges
builder.add_edge(START, "expand_goal")
builder.add_edge("expand_goal", "route_subject")
builder.add_edge("route_subject", "generate_outline")